In [ ]:
import json
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engine" / "vendor" / "control_train.py").exists())
sys.path.insert(0, str(root / "engine" / "vendor"))
from control_train import Config, train

settings = json.loads((root / "engine" / "evidence" / "config.json").read_text())
cfg = Config(**settings)
print(settings)

In [ ]:
prepared = root / "workspace" / "restored" / "sarscope-baseline-20261108" / "prepared"
needed = ["train_db.npy", "train_ctx.npy", "train_lab.npy", "train_support.npy",
          "val_db.npy", "val_ctx.npy", "val_lab.npy", "val_support.npy",
          "stats.json", "manifest.json", "train_meta.json", "val_meta.json"]
missing = [name for name in needed if not (prepared / name).is_file()]
if missing:
    raise FileNotFoundError(f"Extract the prepared dataset to {prepared}. Missing: {missing}")
print('Training data:', prepared)

In [ ]:
run_folder = root / "workspace" / "training_reproduction"
result = train(prepared, run_folder, cfg)
print(result)

In [ ]:
import pandas as pd
history = pd.DataFrame(json.loads((run_folder / "history.json").read_text()))
display(history[["epoch", "train_CE", "validation_CE", "validation_OA", "validation_mIoU"]].tail())